# 건물은 학생을 만드는가 — 분석 재현 노트북

**부산대학교 사회학과 학술제 2026** · 원고: `학술제_최종원고_v8.pdf`

---

## 데이터 리터러시 원칙 (코드를 열기 전에)

1. **개인 단위 원자료는 저장소에 포함하지 않는다.** `data/survey_data.csv`는 원고에 보고된 집계값만 담은 "요약 원본"이다. 개인 응답 88행을 임의로 재구성(시뮬레이션)하는 것은 금지 — 보고된 평균·SD·g·CI는 재현하지만, 지어낸 행 데이터는 없다.
2. **결측치는 채우지 않는다.** 실측 17개 지표 중 3개(주차 정보, 책걸상 훼손률, 층당 휴게 면적)는 미측정으로 보고되었으며, 추정 대체 없이 14개 지표로 종합 점수를 계산했다. AI가 결측을 평균으로 조용히 채우면 표준편차가 줄어드는 왜곡이 생긴다.
3. **정제는 청소가 아니라 판단이다.** 응답 102건 → 14건 제외(사전등록된 규칙: 소속·캠퍼스 불일치, 직선 응답, 중복 응답) → n = 88. 모든 판단은 `survey_data.csv` 상단의 정제 로그에 기록되어 있다.
4. **평균만 들이밀지 않는다.** M(SD), n, Hedges' g, 부트스트랩 95% CI를 함께 보고하고, 분포(이봉성)를 확인한 뒤에야 결론을 내린다.

이 노트북은 Colab/Kaggle 환경에서 `data/survey_data.csv`의 요약 블록을 읽어 원고의 핵심 수치를 재현한다.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from io import StringIO

sns.set_theme(style="darkgrid", palette="deep")

# ---- 요약 CSV 블록 파싱 (# 주석 = 정제 로그, 블록 헤더 행은 'table,...' 시작) ----
def read_blocks(path):
    with open(path, encoding="utf-8") as f:
        lines = [ln.rstrip("\n") for ln in f if not ln.startswith("#") and ln.strip()]
    blocks, header = {}, None
    for ln in lines:
        if ln.startswith("table,"):
            header = ln.split(",")  # 첫 컬럼 'table' = 블록명(domain, hypothesis, ...)
            continue
        parts = ln.split(",")
        if header is None:
            continue
        row = dict(zip(header, parts[: len(header)]))
        key = row.get("table", "_")
        blocks.setdefault(key, []).append(row)
    return {k: pd.DataFrame(v) for k, v in blocks.items()}

blocks = read_blocks("../data/survey_data.csv")
print("블록:", list(blocks.keys()))

In [ ]:
# ---- 발견 1: 실측 상대 점수 (원고 Table 2 / Figure 2 재현) ----
dom = blocks["domain"].copy()
dom[["score_biology", "score_econtrade"]] = dom[["score_biology", "score_econtrade"]].astype(float)
melted = dom.melt(id_vars="domain", value_vars=["score_biology", "score_econtrade"],
                  var_name="building", value_name="score")
melted["building"] = melted["building"].map(
    {"score_biology": "생물관", "score_econtrade": "경제통상관"})

order = ["건물 기반·접근", "학습·휴게 공간", "위생·편의", "종합"]
ax = sns.barplot(data=melted, x="score", y="domain", hue="building", order=order,
                 palette={"생물관": "#DD8452", "경제통상관": "#4C72B0"})
ax.axvline(1.0, color="white", linestyle="--", linewidth=1.2)
ax.set_xlim(0, 1.5)
ax.set_title("실측 종합 상대 점수: 생물관 0.80 vs 경통관 1.20")
plt.show()

assert abs(dom.loc[dom.domain == "종합", "score_biology"].iloc[0] - 0.80) < 1e-9
assert abs(dom.loc[dom.domain == "종합", "score_econtrade"].iloc[0] - 1.20) < 1e-9

In [ ]:
# ---- 발견 2: 가설 검정 g (원고 Table 4 → Figure 3 숲 그림 재현) ----
hyp = blocks["hypothesis"].copy()
for c in ["M_econ", "SD_econ", "M_soc", "SD_soc", "g", "ci_low", "ci_high"]:
    hyp[c] = pd.to_numeric(hyp[c], errors="coerce")

fig, ax = plt.subplots(figsize=(8, 4))
for i, r in hyp.iterrows():
    sig = r.g > 0 and r.ci_low > 0
    ax.plot([r.ci_low, r.ci_high], [i, i],
            color="white" if sig else "#6e6e73", lw=2.5)
    ax.plot([r.g], [i], "o", ms=9, color="white" if sig else "#3a3a3c")
ax.axvline(0, color="white", lw=1.4)
ax.set_yticks(range(len(hyp)))
ax.set_yticklabels(hyp.outcome)
ax.set_xlabel("Hedges' g (양수 = 경통관 높음)")
ax.set_title("시설 만족 g=1.41 vs 대학생활 g=0.15, 삶의 만족 g=0.36")
plt.show()

assert hyp.loc[hyp.outcome.str.contains("시설"), "g"].iloc[0] == 1.41
assert hyp.loc[hyp.outcome.str.contains("대학생활"), "g"].iloc[0] == 0.15

In [ ]:
# ---- 발견 4: 학년별 하위집단 (원고 Figure 5 재현) — 이봉 구조의 출처 ----
yg = blocks["yeargroup"].copy()
points = yg[yg.subgroup != "이중차분(DiD)"].copy()
points["mean"] = pd.to_numeric(points["mean"])
points["n"] = pd.to_numeric(points["n"], errors="coerce")
points["소속"] = points.subgroup.str.split(" ").str[0]
points["학년"] = points.subgroup.str.split(" ").str[1]

g = sns.catplot(data=points, x="학년", y="mean", hue="소속", col="outcome", kind="point",
                palette={"경통관": "#4C72B0", "사회대": "#DD8452"},
                height=4, aspect=1.1, order=["1학년", "2~4학년"])
g.set_axis_labels("", "평균 (1~7점)")
g.fig.subplots_adjust(top=0.82)
g.fig.suptitle("2~4학년: 삶의 만족 g=0.72 (p=.006) — 1학년은 차이 없음")
plt.show()

# 이중차분 값 확인
did = yg[yg.subgroup == "이중차분(DiD)"]
assert did.loc[did.outcome == "삶의 만족", "mean"].astype(float).iloc[0] == 1.33

In [ ]:
# ---- 결론: 새 사회관 희망 시설 (원고 Table 7 재현) ----
needs = blocks["needs"].copy()
needs["pct"] = pd.to_numeric(needs["pct"])
new = needs[needs.group == "새 사회관 희망"].sort_values("pct", ascending=False)
ax = sns.barplot(data=new, x="pct", y="facility", color="white")
ax.set_xlim(0, 60)
for i, r in enumerate(new.itertuples()):
    ax.text(r.pct + 1, i, f"{r.pct:.0f}% (n={r.n})", va="center", color="white")
ax.set_title("새 사회관 희망 시설 (사회대 n=48, 최대 3개 선택)")
plt.show()

## 정제 로그 요약 & 한계

- **정제 로그**: `data/survey_data.csv` 상단 주석 참조. 응답 102 → 88(경통 40 / 사회 48), 미측정 3지표 대체 없음, 직선·중복 응답 없음.
- **검정력**: 목표(집단당 65명, d=0.50)에 미달 — 본 분석은 d=0.61 이상을 80% 검정력으로 잡는다. 유의하지 않은 결과는 '차이가 없다'가 아니라 '충분한 차이를 못 잡았을 수 있음'으로 읽는다.
- **탐색 분석**: 학년별·이봉·매개 결과는 탐색적이며 다중비교 보정 없음. 방향은 신뢰하되 크기는 확인 필요.
- **블라인드 사진(H4)**: 건물마다 다른 면 4장씩 — 건물 차이와 면 차이가 혼재 가능. 응답자 72%가 4장 이상에서 건물을 알아봤다고 응답(원고 본문 참조).
- **반응성**: 이 노트북의 코드는 요약 통계의 재현에 한정되며, 개인 단위 원자료 없이 p값·분산 분석을 재계산할 수는 없다.